# Time-Series Embedding — GPU Cluster (DDP) Training

Companion to `contrastive_training_starter.ipynb`, scaled for **>3M accounts**
on a **GPU cluster** with a **CPU-/RAM-constrained** data path.

### Why this notebook *orchestrates* instead of training inline
DDP needs one process per GPU, launched by `torchrun`. Spawning that inside a
Jupyter kernel is fragile (fork/CUDA re-init, hung NCCL groups). The robust
pattern — and what this notebook does — is:

1. (Optionally) convert your parquet/tensor frame to memmap files **in
   chunks**, so you never hold 28 GB in RAM.
2. Build the `torchrun` command for your cluster topology.
3. Launch it as a subprocess and **stream logs back into the notebook**.
4. Extract embeddings in chunks afterward.

### How the CPU/RAM concern is handled
`scripts/train_ddp.py --streaming` uses `ChunkedIterableDataset`: it reads
**contiguous chunks** from the memmap, yields the samples, then `del`s the
chunk so its RAM is reclaimed **before the next batch is read**. The training
loop also drops every batch's GPU/pinned tensors per step and can call
`torch.cuda.empty_cache()` periodically. Peak host memory per worker is bounded
by `chunk_size`, not by N — so 3M or 30M rows use the same RAM.

In [ ]:
import os, sys, json, shlex, subprocess, pathlib, textwrap

REPO_ROOT = pathlib.Path.cwd()
if not (REPO_ROOT / 'ts_embed').exists():
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
print('repo root:', REPO_ROOT)

## 1. Detect cluster topology

Works for a single multi-GPU box (`--standalone`) or a Slurm multi-node job.
If you use a different scheduler, set these env vars before launching the
notebook / edit the cell.

In [ ]:
import torch

SLURM = 'SLURM_JOB_ID' in os.environ
if SLURM:
    NNODES = int(os.environ.get('SLURM_NNODES', '1'))
    GPUS_PER_NODE = int(os.environ.get('SLURM_GPUS_ON_NODE',
                        str(torch.cuda.device_count())))
    NODE_RANK = int(os.environ.get('SLURM_NODEID', '0'))
    MASTER_ADDR = os.environ.get('MASTER_ADDR', '127.0.0.1')
    MASTER_PORT = os.environ.get('MASTER_PORT', '29500')
    RDZV_ID = os.environ['SLURM_JOB_ID']
else:
    NNODES = 1
    GPUS_PER_NODE = max(1, torch.cuda.device_count())
    NODE_RANK = 0
    MASTER_ADDR, MASTER_PORT, RDZV_ID = '127.0.0.1', '29500', 'standalone'

WORLD_SIZE = NNODES * GPUS_PER_NODE
print(f'nodes={NNODES} gpus/node={GPUS_PER_NODE} world_size={WORLD_SIZE} '
      f'node_rank={NODE_RANK} slurm={SLURM}')
assert torch.cuda.is_available(), 'No CUDA devices visible'

## 2. (Optional) Chunked parquet → memmap conversion

Run this **once** during data prep. It streams the parquet/tensor frame in
row chunks and writes directly into preallocated `np.memmap` files, so host
RAM stays at ~one chunk regardless of the 3M+ total. **Skip this cell if you
already built `numeric.npy / missing.npy / categorical.npy`.**

Adapt the `read_chunk` function to your actual source (parquet partitions,
Arrow dataset, a tensor frame's `.iloc` slices, etc.). Expected per-row
layout after reshaping: `(24, 98)` numeric, `(24, 98)` missing, `(24, 2)` cat.

In [ ]:
import numpy as np

DO_CONVERT = False          # flip to True to run conversion
N_TOTAL    = 3_000_000      # number of accounts
T, F_NUM, F_CAT = 24, 98, 2
CHUNK = 50_000              # rows per chunk -> RAM ~ CHUNK*T*F*4 bytes

data_dir = REPO_ROOT / 'data'
data_dir.mkdir(exist_ok=True)
p_num = data_dir / 'numeric.npy'
p_mis = data_dir / 'missing.npy'
p_cat = data_dir / 'categorical.npy'

def read_chunk(start, end):
    """REPLACE ME. Return (numeric, missing, categorical) for rows [start:end).
    Shapes: (k,T,F_NUM) float32, (k,T,F_NUM) uint8, (k,T,F_CAT) int8."""
    k = end - start
    return (np.random.randn(k, T, F_NUM).astype(np.float32),
            (np.random.rand(k, T, F_NUM) < 0.1).astype(np.uint8),
            (np.random.rand(k, T, F_CAT) < 0.5).astype(np.int8))

if DO_CONVERT:
    mm_num = np.lib.format.open_memmap(p_num, 'w+', np.float32, (N_TOTAL, T, F_NUM))
    mm_mis = np.lib.format.open_memmap(p_mis, 'w+', np.uint8,   (N_TOTAL, T, F_NUM))
    mm_cat = np.lib.format.open_memmap(p_cat, 'w+', np.int8,    (N_TOTAL, T, F_CAT))
    for s in range(0, N_TOTAL, CHUNK):
        e = min(s + CHUNK, N_TOTAL)
        bn, bm, bc = read_chunk(s, e)
        mm_num[s:e], mm_mis[s:e], mm_cat[s:e] = bn, bm, bc
        del bn, bm, bc                       # release the chunk immediately
        if (s // CHUNK) % 10 == 0:
            print(f'  wrote rows {s:,}/{N_TOTAL:,}', flush=True)
    mm_num.flush(); mm_mis.flush(); mm_cat.flush()
    del mm_num, mm_mis, mm_cat
    print('conversion done')
else:
    print('skipped — assuming memmap files already exist at', data_dir)

## 3. Build the training command

Key memory knobs passed to `scripts/train_ddp.py`:

| flag | effect |
|---|---|
| `--streaming` | use `ChunkedIterableDataset` (contiguous reads, chunk freed per batch) |
| `--chunk-size` | rows held in RAM per worker; lower = less RAM, more I/O |
| `--num-workers` | fewer workers = less aggregate page-cache pressure |
| `--prefetch-factor` | batches prefetched per worker; 1–2 caps staging RAM |
| `--empty-cache-every` | periodic `torch.cuda.empty_cache()` to curb fragmentation |

`--batch-size` is **per GPU**; effective batch = `batch_size × world_size`.

In [ ]:
RUN_DIR = REPO_ROOT / 'runs' / 'ddp_streaming'

train_args = [
    '--numeric', str(p_num), '--missing', str(p_mis), '--categorical', str(p_cat),
    '--out', str(RUN_DIR),
    '--streaming',
    '--chunk-size', '4096',
    '--num-workers', '4',
    '--prefetch-factor', '2',
    '--empty-cache-every', '500',
    '--batch-size', '512',
    '--epochs', '30',
    '--d-model', '192', '--n-layers', '4', '--n-heads', '6', '--proj-dim', '256',
    '--time-mask-prob', '0.25', '--feature-mask-prob', '0.30',
    '--lr', '1e-3', '--warmup-steps', '2000',
    '--log-every', '50', '--ckpt-every', '1',
]

if SLURM and NNODES > 1:
    rdzv = ['--nnodes', str(NNODES), '--node_rank', str(NODE_RANK),
            '--rdzv_id', RDZV_ID, '--rdzv_backend', 'c10d',
            '--rdzv_endpoint', f'{MASTER_ADDR}:{MASTER_PORT}']
else:
    rdzv = ['--standalone', '--nnodes', '1']

cmd = ['torchrun', *rdzv, '--nproc_per_node', str(GPUS_PER_NODE),
       'scripts/train_ddp.py', *train_args]
print(' \\\n  '.join(shlex.quote(c) for c in cmd))

## 4. Launch and stream logs

On a multi-node Slurm job, run this notebook (or the equivalent cell) **once
per node** — each invocation launches that node's `torchrun`, and the c10d
rendezvous joins them. Interrupting the kernel terminates the child process
group cleanly.

In [ ]:
import signal
RUN_DIR.mkdir(parents=True, exist_ok=True)
log_path = RUN_DIR / f'train_node{NODE_RANK}.log'

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env.setdefault('OMP_NUM_THREADS', '4')           # avoid CPU oversubscription
env.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

proc = subprocess.Popen(
    cmd, cwd=str(REPO_ROOT), env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, start_new_session=True,
)
try:
    with open(log_path, 'w') as lf:
        for line in proc.stdout:
            print(line, end='')
            lf.write(line)
    proc.wait()
    print(f'\n[exit code {proc.returncode}]')
except KeyboardInterrupt:
    os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
    print('\n[interrupted — sent SIGTERM to the training process group]')

## 5. Extract embeddings (chunked, memory-bounded)

`scripts/extract_embeddings.py` streams the dataset and writes a memmapped
`(N, d_model)` output, so producing 3M embeddings also stays within bounded
RAM. Run on a **single GPU** (no DDP needed for inference).

In [ ]:
import glob
ckpts = sorted(glob.glob(str(RUN_DIR / 'ckpt_epoch*.pt')))
assert ckpts, 'no checkpoints found yet — let training run first'
latest = ckpts[-1]
print('using checkpoint:', latest)

extract_cmd = [
    sys.executable, 'scripts/extract_embeddings.py',
    '--ckpt', latest,
    '--numeric', str(p_num), '--missing', str(p_mis), '--categorical', str(p_cat),
    '--out', str(RUN_DIR / 'embeddings.npy'),
    '--batch-size', '4096', '--num-workers', '4',
]
print(subprocess.run(extract_cmd, cwd=str(REPO_ROOT),
                     capture_output=True, text=True).stdout)

In [ ]:
# Collapse sanity-check on a memmapped sample (don't load all 3M into RAM).
emb = np.load(RUN_DIR / 'embeddings.npy', mmap_mode='r')
sample = np.asarray(emb[np.random.choice(emb.shape[0],
                    size=min(50_000, emb.shape[0]), replace=False)])
std = sample.std(axis=0)
print(f'embeddings shape={emb.shape}  per-dim std mean={std.mean():.4f} '
      f'min={std.min():.4f}')
assert std.mean() > 1e-2, 'COLLAPSE: raise var_coef / lower LR / reduce mask prob'
print('OK — embeddings carry variance, no collapse detected')

## Operational notes

- **RAM tuning order**: if a node still pressures memory, lower `--chunk-size`
  first, then `--num-workers`, then `--prefetch-factor` (to 1).
- **Throughput vs RAM**: smaller chunks = more, smaller I/O reads. On fast
  NVMe a 4096-row chunk is a good balance; on networked storage go larger.
- **VICReg across ranks**: the DDP script already gathers projections across
  GPUs so the variance/covariance (anti-collapse) stats use the full effective
  batch — important at large `world_size`.
- **Resuming**: checkpoints land in `RUN_DIR`; load `ckpt['model']` (strip the
  `module.` / `_orig_mod.` prefixes as `extract_embeddings.py` does).
- **Multi-node**: launch this notebook once per node with the Slurm env vars
  set; the c10d rendezvous (`--rdzv_endpoint`) joins the ranks.